In [2]:
import numpy as np
import scipy.io as sio
from scipy.io import savemat
import scipy.signal as sp
import matplotlib.pyplot as plt
import math as mt
import os.path as op
import obspy
import statistics as st
from obspy.core import read
from obspy.core.trace import Trace
from obspy.core.stream import Stream
from obspy import UTCDateTime
from obspy import read_inventory
from scipy.fft import fft, fftfreq
import datetime
from obspy.clients.filesystem.sds import Client

In [7]:
# for the borehole station
from obspy import read_inventory
from obspy.core.inventory import Inventory, Network, Station, Channel
from obspy.core.inventory.response import Response, ResponseStage, InstrumentSensitivity

# ----------------------------
# Load generic sensor XML
# ----------------------------
inv_generic = read_inventory("/data/gravwav/koley/HRBS/NL.HRBS.xml")
sensor_response = inv_generic[0][0][0].response  # first channel

# ----------------------------
# Define custom station info
# ----------------------------
network_code = "NL"
station_code = "HRBS"
latitude = 10.0
longitude = 20.0
elevation = 100.0
location_code = "01"  # empty to match traces with empty location
channels = ["BHZ", "BH1", "BH2"]

# ----------------------------
# Digitizer parameters
# ----------------------------
digitizer_gain = 2**24 / 40.0*1000  # replace with actual ADC spec
waveform_in_counts = True      # set False if waveform is already in volts

# ----------------------------
# Create channels with responses
# ----------------------------
channel_objects = []

for ch in channels:
    full_response = Response()

    # Stage 1: Sensor stage (m/s -> V), gain included in XML
    full_response.response_stages.append(sensor_response.response_stages[0])

    # Stage 2: Digitizer stage (V -> counts), only if waveform is in counts
    if waveform_in_counts:
        digitizer_stage = ResponseStage(
            stage_sequence_number=2,
            stage_gain=digitizer_gain,
            stage_gain_frequency=1.0,
            input_units="V",
            output_units="counts"
        )
        digitizer_stage.stage_type = "A/D conversion"
        full_response.response_stages.append(digitizer_stage)

    # ----------------------------
    # Set top-level InstrumentSensitivity
    # ----------------------------
    full_response.instrument_sensitivity = InstrumentSensitivity(
        value=1.0,  # gain already included in stages
        frequency=1.0,
        input_units=full_response.response_stages[0].input_units,
        output_units=full_response.response_stages[-1].output_units
    )

    # Create the channel and attach response
    channel = Channel(
        code=ch,
        location_code=location_code,
        latitude=latitude,
        longitude=longitude,
        elevation=elevation,
        depth=0.0,
        response=full_response,
        start_date=None,   # Always active
        end_date=None      # Always active
    )
    channel_objects.append(channel)

# ----------------------------
# Create Station and Network
# ----------------------------
station = Station(
    code=station_code,
    latitude=latitude,
    longitude=longitude,
    elevation=elevation,
    channels=channel_objects,
    start_date=None,  # Always active
    end_date=None
)

network = Network(network_code, stations=[station])
custom_inventory = Inventory(networks=[network], source="User")

# ----------------------------
# Write StationXML
# ----------------------------
custom_inventory.write("/data/gravwav/koley/HRBS/myHRBS_final.xml", format="STATIONXML")

In [3]:
respPath = '/data/gravwav/koley/HRBS/myHRBS_final.xml';
invNow = read_inventory(respPath)
ch = invNow[0][0][0]  # first network, first station, first channel
print(ch.response)

Channel Response
	From m/s (None) to counts (None)
	Overall Sensitivity: 1 defined at 1.000 Hz
	2 stages:
		Stage 1: PolesZerosResponseStage from m/s to V, gain: 754.3
		Stage 2: ResponseStage from V to V, gain: 4.1943e+08


In [ ]:
# for the borehole station
from obspy import read_inventory
from obspy.core.inventory import Inventory, Network, Station, Channel
from obspy.core.inventory.response import Response, ResponseStage, InstrumentSensitivity

# ----------------------------
# Load generic sensor XML
# ----------------------------
inv_generic = read_inventory("/data/gravwav/koley/HRBS/NL.HRBS.xml")
sensor_response = inv_generic[0][0][0].response  # first channel

# ----------------------------
# Define custom station info
# ----------------------------
network_code = "NL"
station_code = "HRBS"
latitude = 10.0
longitude = 20.0
elevation = 100.0
location_code = "01"  # empty to match traces with empty location
channels = ["BHZ", "BH1", "BH2"]

# ----------------------------
# Digitizer parameters
# ----------------------------
digitizer_gain = 2**24 / 40.0*1000  # replace with actual ADC spec
waveform_in_counts = True      # set False if waveform is already in volts

# ----------------------------
# Create channels with responses
# ----------------------------
channel_objects = []

for ch in channels:
    full_response = Response()

    # Stage 1: Sensor stage (m/s -> V), gain included in XML
    full_response.response_stages.append(sensor_response.response_stages[0])

    # Stage 2: Digitizer stage (V -> counts), only if waveform is in counts
    if waveform_in_counts:
        digitizer_stage = ResponseStage(
            stage_sequence_number=2,
            stage_gain=digitizer_gain,
            stage_gain_frequency=1.0,
            input_units="V",
            output_units="counts"
        )
        digitizer_stage.stage_type = "A/D conversion"
        full_response.response_stages.append(digitizer_stage)

    # ----------------------------
    # Set top-level InstrumentSensitivity
    # ----------------------------
    full_response.instrument_sensitivity = InstrumentSensitivity(
        value=1.0,  # gain already included in stages
        frequency=1.0,
        input_units=full_response.response_stages[0].input_units,
        output_units=full_response.response_stages[-1].output_units
    )

    # Create the channel and attach response
    channel = Channel(
        code=ch,
        location_code=location_code,
        latitude=latitude,
        longitude=longitude,
        elevation=elevation,
        depth=0.0,
        response=full_response,
        start_date=None,   # Always active
        end_date=None      # Always active
    )
    channel_objects.append(channel)

# ----------------------------
# Create Station and Network
# ----------------------------
station = Station(
    code=station_code,
    latitude=latitude,
    longitude=longitude,
    elevation=elevation,
    channels=channel_objects,
    start_date=None,  # Always active
    end_date=None
)

network = Network(network_code, stations=[station])
custom_inventory = Inventory(networks=[network], source="User")

# ----------------------------
# Write StationXML
# ----------------------------
custom_inventory.write("/data/gravwav/koley/HRBS/myHRBS_surf.xml", format="STATIONXML")